# 05 — Inventory Prediction

**AI Financial Intelligence Platform — ML Module**

---

## Purpose

Predict future inventory demand to prevent stockouts and overstock situations.

### Goals
- Predict quantity_on_hand per product per warehouse
- Trigger reorder alerts when predicted stock falls below reorder_level
- Estimate optimal reorder quantities

### Models to Evaluate
- XGBoost demand model
- Simple exponential smoothing (SES)
- Moving average baseline

### Sections
1. Load inventory + stock_movements + sales
2. Compute consumption rate per product
3. Feature engineering
4. Model training
5. Reorder point prediction
6. Evaluation
7. Save model


In [ ]:
import os
import json
import pickle
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
import xgboost as xgb

warnings.filterwarnings('ignore')

DATA_DIR    = '../datasets/synthetic/'
MODELS_DIR  = '../models/'
REPORTS_DIR = '../reports/'
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

## 1. Load Inventory + Stock Movements + Sales


In [ ]:
# Load tables
inventory       = pd.read_csv(DATA_DIR + 'inventory.csv')
stock_movements = pd.read_csv(DATA_DIR + 'stock_movements.csv')
products        = pd.read_csv(DATA_DIR + 'products.csv')
warehouses      = pd.read_csv(DATA_DIR + 'warehouses.csv')
sale_items      = pd.read_csv(DATA_DIR + 'sale_items.csv')
sales           = pd.read_csv(DATA_DIR + 'sales.csv')
categories      = pd.read_csv(DATA_DIR + 'categories.csv')

# Parse dates
stock_movements['moved_at'] = pd.to_datetime(stock_movements['moved_at'], errors='coerce')
sales['sale_date']          = pd.to_datetime(sales['sale_date'], errors='coerce')

# ── Full inventory view ───────────────────────────────────────────────────────
inv_full = (
    inventory
    .merge(products[['id', 'name', 'sku', 'category_id', 'cost_price']],
           left_on='product_id', right_on='id')
    .merge(categories[['id', 'name']], left_on='category_id', right_on='id',
           suffixes=('_prod', '_cat'))
    .merge(warehouses[['id', 'name']], left_on='warehouse_id', right_on='id',
           suffixes=('_cat', '_wh'))
)

inv_full['stock_value']  = inv_full['quantity_on_hand'] * inv_full['cost_price']
inv_full['is_low_stock'] = inv_full['quantity_on_hand'] <= inv_full['reorder_level']

print('Inventory full shape:', inv_full.shape)
print(f'Low-stock SKUs: {inv_full["is_low_stock"].sum()} / {len(inv_full)}')
print(f'Total stock value: {inv_full["stock_value"].sum():,.0f}')

# ── Monthly outbound movements (demand proxy) ─────────────────────────────────
outbound = stock_movements[stock_movements['movement_type'] == 'out'].copy()
outbound = outbound.dropna(subset=['moved_at'])

monthly_demand = (
    outbound
    .groupby(['product_id', pd.Grouper(key='moved_at', freq='ME')])
    .agg(units_out=('quantity', 'sum'))
    .reset_index()
    .rename(columns={'moved_at': 'month'})
    .sort_values(['product_id', 'month'])
    .reset_index(drop=True)
)

print('\nMonthly demand (outbound) shape:', monthly_demand.shape)

## 2. Compute Consumption Rate


In [ ]:
# ── Average weekly consumption per product ────────────────────────────────────
# Sum total outbound units / number of weeks in the dataset
date_range_weeks = (
    (stock_movements['moved_at'].max() - stock_movements['moved_at'].min()).days / 7
)
date_range_weeks = max(date_range_weeks, 1)

consumption = (
    outbound
    .groupby('product_id')['quantity']
    .sum()
    .reset_index()
    .rename(columns={'quantity': 'total_outbound'})
)
consumption['avg_weekly_demand'] = consumption['total_outbound'] / date_range_weeks
consumption['avg_daily_demand']  = consumption['avg_weekly_demand'] / 7

# Merge with inventory to compute days-of-stock-remaining
inv_consumption = inv_full.merge(consumption, on='product_id', how='left')
inv_consumption['avg_daily_demand'] = inv_consumption['avg_daily_demand'].fillna(0.01)
inv_consumption['days_of_stock_remaining'] = (
    inv_consumption['quantity_on_hand'] / inv_consumption['avg_daily_demand']
).clip(upper=365)

print('Consumption rate stats:')
print(inv_consumption[['sku', 'quantity_on_hand', 'avg_daily_demand',
                        'days_of_stock_remaining', 'reorder_level']]
      .sort_values('days_of_stock_remaining')
      .head(10)
      .to_string(index=False))

## 3. Feature Engineering


In [ ]:
# ── Build feature matrix from monthly_demand ──────────────────────────────────
def add_inventory_features(df: pd.DataFrame,
                             group_col: str = 'product_id',
                             target: str = 'units_out') -> pd.DataFrame:
    d = df.copy().sort_values([group_col, 'month'])

    # Lag demand
    for lag in [1, 2, 3, 6]:
        d[f'lag_{lag}'] = d.groupby(group_col)[target].shift(lag)

    # Rolling stats
    d['roll3_mean'] = (
        d.groupby(group_col)[target]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )
    d['roll6_mean'] = (
        d.groupby(group_col)[target]
        .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
    )

    # Calendar
    d['month_num'] = pd.to_datetime(d['month']).dt.month
    d['quarter']   = pd.to_datetime(d['month']).dt.quarter
    d['year']      = pd.to_datetime(d['month']).dt.year

    # Seasonal flags
    d['is_q4']    = (d['quarter'] == 4).astype(int)
    d['is_q1']    = (d['quarter'] == 1).astype(int)

    return d

demand_feat = add_inventory_features(monthly_demand)
demand_feat = demand_feat.dropna(subset=['lag_1', 'lag_2', 'lag_3']).copy()

FEAT_COLS = ['lag_1', 'lag_2', 'lag_3', 'lag_6',
             'roll3_mean', 'roll6_mean',
             'month_num', 'quarter', 'year', 'is_q4', 'is_q1']

print('Demand feature matrix shape:', demand_feat.shape)
print('Features:', FEAT_COLS)

## 4. Model Training


In [ ]:
# Chronological 80/20 split
all_months  = sorted(demand_feat['month'].unique())
split_idx   = int(len(all_months) * 0.80)
split_month = all_months[split_idx]

train_df = demand_feat[demand_feat['month'] <  split_month].copy()
test_df  = demand_feat[demand_feat['month'] >= split_month].copy()

X_tr = train_df[FEAT_COLS].fillna(0).values
y_tr = train_df['units_out'].values
X_te = test_df[FEAT_COLS].fillna(0).values
y_te = test_df['units_out'].values

# ── Model A: Moving Average Baseline ─────────────────────────────────────────
ma_pred = test_df['roll3_mean'].fillna(test_df['lag_1'].fillna(0)).values

# ── Model B: XGBoost ─────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_tr, y_tr)
xgb_pred = xgb_model.predict(X_te)
xgb_pred = np.maximum(xgb_pred, 0)  # Demand cannot be negative

print(f'Split month: {split_month}')
print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')

## 5. Reorder Point Prediction


In [ ]:
# ── Predicted demand for the next period ─────────────────────────────────────
# Use the most recent record per product to predict next-month demand
latest_feat = (
    demand_feat
    .sort_values('month')
    .groupby('product_id')
    .last()
    .reset_index()
)

X_next = latest_feat[FEAT_COLS].fillna(0).values
latest_feat['predicted_demand_next_month'] = np.maximum(
    xgb_model.predict(X_next), 0
)

# ── Merge with inventory snapshot ─────────────────────────────────────────────
reorder_df = (
    inv_consumption[['product_id', 'sku', 'name_prod', 'quantity_on_hand',
                     'reorder_level', 'avg_daily_demand', 'days_of_stock_remaining']]
    .merge(latest_feat[['product_id', 'predicted_demand_next_month']], on='product_id', how='left')
)

# Lead time = 7 days (assumed); safety stock = 1.5x daily demand × lead time
LEAD_TIME_DAYS = 7
reorder_df['safety_stock']     = reorder_df['avg_daily_demand'] * LEAD_TIME_DAYS * 1.5
reorder_df['reorder_point']    = (
    reorder_df['avg_daily_demand'] * LEAD_TIME_DAYS + reorder_df['safety_stock']
)
reorder_df['needs_reorder']    = (
    reorder_df['quantity_on_hand'] <= reorder_df['reorder_point']
).astype(int)
reorder_df['suggested_order_qty'] = (
    reorder_df['predicted_demand_next_month'] + reorder_df['safety_stock'] -
    reorder_df['quantity_on_hand']
).clip(lower=0).round(0)

alert_items = reorder_df[reorder_df['needs_reorder'] == 1].sort_values('days_of_stock_remaining')
print(f'Products needing reorder: {len(alert_items)} / {len(reorder_df)}')
print(alert_items[['sku', 'name_prod', 'quantity_on_hand', 'reorder_point',
                    'days_of_stock_remaining', 'suggested_order_qty']]
      .head(15)
      .to_string(index=False))

# ── Visualise reorder alerts ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
top_alerts = alert_items.head(15)
colors = ['tomato' if r <= 3 else 'orange' for r in top_alerts['days_of_stock_remaining']]
ax.barh(top_alerts['sku'], top_alerts['days_of_stock_remaining'], color=colors)
ax.axvline(LEAD_TIME_DAYS, color='red', ls='--', label=f'Lead time ({LEAD_TIME_DAYS}d)')
ax.set_title('Days of Stock Remaining — Products Requiring Reorder', fontweight='bold')
ax.set_xlabel('Days of Stock Remaining')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

## 6. Evaluation


In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Compare models
models_eval = [
    ('Moving Average', ma_pred),
    ('XGBoost',        xgb_pred),
]

rows = []
for name, preds in models_eval:
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    mae  = mean_absolute_error(y_te, preds)
    mp   = mape(y_te, preds)
    rows.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'MAPE_%': mp})
    print(f'{name:<20}  RMSE={rmse:>8.2f}  MAE={mae:>8.2f}  MAPE={mp:>6.2f}%')

eval_df = pd.DataFrame(rows)

# Stockout rate on test set
xgb_preds_series = pd.Series(xgb_pred, index=test_df.index)
actual_series    = pd.Series(y_te,     index=test_df.index)
stockout_rate    = (actual_series > xgb_preds_series).mean() * 100
overstock_rate   = (actual_series < xgb_preds_series * 0.5).mean() * 100
print(f'\nStockout rate (actual > predicted): {stockout_rate:.1f}%')
print(f'Overstock rate (actual < 50%% of predicted): {overstock_rate:.1f}%')

# Save report
report = {
    'task': 'inventory_prediction',
    'models': rows,
    'stockout_rate_pct': stockout_rate,
    'overstock_rate_pct': overstock_rate,
    'reorder_alerts': len(alert_items),
}
with open(os.path.join(REPORTS_DIR, 'inventory_prediction_eval.json'), 'w') as f:
    json.dump(report, f, indent=2, default=float)
print('Evaluation report saved.')

## 7. Save Model


In [ ]:
# Save XGBoost demand model
model_path = os.path.join(MODELS_DIR, 'inventory_demand_xgboost.pkl')
with open(model_path, 'wb') as f:
    pickle.dump({'model': xgb_model, 'feature_cols': FEAT_COLS}, f)
print(f'XGBoost inventory model saved → {model_path}')

# Save reorder alerts snapshot as CSV for downstream use
alert_path = os.path.join(MODELS_DIR, 'reorder_alerts.csv')
reorder_df.to_csv(alert_path, index=False)
print(f'Reorder alerts snapshot saved → {alert_path}')

# Feature metadata
with open(os.path.join(MODELS_DIR, 'inventory_feature_meta.json'), 'w') as f:
    json.dump({'task': 'inventory_prediction', 'features': FEAT_COLS}, f, indent=2)
print('Feature metadata saved.')